In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import sys
sys.path.append('../backend')

from model_loader import load_model
from preprocessing import preprocess_input

c:\Users\CLL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.6.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
df = pd.read_csv(r'C:\Users\CLL\OneDrive\Documents\GitHub\F1-Race-Predictor\notebooks\2015_to_2025_df.csv')


In [4]:
REG_DISRUPTION = {
    2014: 5,  # V8 â†’ V6 turbo-hybrid (MGU-H, MGU-K, ERS). Launched 7yr Mercedes dominance.
    2015: 0, 2016: 0,
    2017: 2,  # Wider cars/tyres, more downforce. Same PU formula.
    2018: 0,
    2019: 1,  # Simplified front wing, dirty air reduction tweaks.
    2020: 0,  # COVID freeze.
    2021: 1,  # Floor cuts, cost cap intro. Procedural more than technical.
    2022: 4,  # Full ground-effect overhaul. Ended Mercedes era.
    2023: 0, 2024: 0, 2025: 0,
    2026: 5,  # New PU (MGU-H removed, 350kW MGU-K), active aero, 30kg lighter.
}

In [7]:
MAJOR_RESET_YEARS = [2014, 2022, 2026]

def add_reg_features(df):
    df = df.copy()
    df["reg_disruption_index"] = df["Season"].map(lambda yr: REG_DISRUPTION.get(yr, 0))
    def years_since(season):
        past_resets = [yr for yr in MAJOR_RESET_YEARS if yr <= season]
        return season - (max(past_resets) if past_resets else 2013)
    df["years_since_reg_change"] = df["Season"].map(years_since)
    return df

df = add_reg_features(df)

## Data Cleaning

Replace sentinel values (`9999`, `0.0`) with `NaN`. Add binary Q2/Q3 appearance flags. Sort chronologically.

In [ ]:
# Replace sentinels
for col in ["Q1_s", "Q2_s", "Q3_s"]:
    df[col] = df[col].replace({9999.0: np.nan, 0.0: np.nan})

# Drop rows with no Q times AND no grid position (DNS/data gap)
all_null_q = df[["Q1_s", "Q2_s", "Q3_s"]].isna().all(axis=1)
df = df[~(all_null_q & df["GridPosition"].isna())].copy()

# Appearance flags
df["Q2_appearance"] = df["Q2_s"].notna().astype(int)
df["Q3_appearance"] = df["Q3_s"].notna().astype(int)

# Must sort chronologically before any rolling/cumulative operation
df = df.sort_values(["Season", "Round"]).reset_index(drop=True)
print(f"Rows after cleaning: {len(df)}")

## Within-Session Qualifying Gaps

Gap to session fastest, computed **within each session independently** — Q1 gap uses only Q1 times, Q2 uses Q2 only, etc. Non-participants get `NaN` (handled natively by XGB/LGBM/CatBoost).

In [ ]:
for q_col, gap_col in [("Q1_s","Q1_gap"), ("Q2_s","Q2_gap"), ("Q3_s","Q3_gap")]:
    session_min = df.groupby(["Season","Round"])[q_col].transform("min")
    df[gap_col] = df[q_col] - session_min

## Rolling & Historical Features

All features are **shifted by 1** (`.shift(1)`) so the current race's own value is never included — prevents data leakage. Cold-start rows receive `NaN`.

In [ ]:
# Season stage: 0→1 ratio across the calendar
rounds_per_season = df.groupby("Season")["Round"].transform("max")
df["season_stage_ratio"] = df["Round"] / rounds_per_season

# Driver career races: count of prior starts (0 = debut)
df["driver_career_races"] = df.groupby("Abbreviation").cumcount()

# Cumulative Q3 appearance rate within season (shifted — excludes current race)
df["driver_q3_rate_season"] = (
    df.groupby(["Season","Abbreviation"])["Q3_appearance"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
# Round 1 NaN → fall back to career rate up to that point
career_q3 = df.groupby("Abbreviation")["Q3_appearance"].transform(
    lambda x: x.shift(1).expanding().mean()
)
df["driver_q3_rate_season"] = df["driver_q3_rate_season"].fillna(career_q3)

In [ ]:
# Driver's historical avg qualifying position at this circuit (no leakage)
df["driver_circuit_q_pos_hist"] = (
    df.groupby(["Abbreviation","EventName"])["GridPosition"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Prior year qualifying position at same circuit
prev = (
    df[["Season","EventName","Abbreviation","GridPosition"]]
    .rename(columns={"GridPosition": "prior_year_q_pos_same_circuit"})
    .assign(Season=lambda x: x["Season"] + 1)
)
df = df.merge(prev, on=["Season","EventName","Abbreviation"], how="left")

In [ ]:
# Team rolling 5-race avg qualifying position (shifted)
df["team_rolling_q_pos_5r"] = (
    df.groupby("TeamName")["GridPosition"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

# Driver rolling 5-race avg race finishing position
df["driver_rolling_race_pos_5r"] = (
    df.groupby("Abbreviation")["Position"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

In [ ]:
# Teammate qualifying gap: avg of (driver GridPos - teammate GridPos) per past race
teammate = (
    df[["Season","Round","TeamName","Abbreviation","GridPosition"]]
    .merge(
        df[["Season","Round","TeamName","Abbreviation","GridPosition"]]
        .rename(columns={"Abbreviation":"tm_abbr","GridPosition":"tm_grid_pos"}),
        on=["Season","Round","TeamName"]
    )
)
teammate = teammate[teammate["Abbreviation"] != teammate["tm_abbr"]].copy()
teammate["q_gap_vs_tm"] = teammate["GridPosition"] - teammate["tm_grid_pos"]
teammate = teammate.sort_values(["Season","Round"])
teammate["teammate_q_gap_season"] = (
    teammate.groupby(["Season","Abbreviation"])["q_gap_vs_tm"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
df = df.merge(
    teammate[["Season","Round","Abbreviation","teammate_q_gap_season"]].drop_duplicates(),
    on=["Season","Round","Abbreviation"], how="left"
)

## Static Lookup Features

Hardcoded from F1 domain knowledge. These add signal beyond `EventName` alone by grouping circuits into meaningful categories.

In [ ]:
# Night races (circuit is artificially lit for the session)
NIGHT_RACES = {
    "Bahrain Grand Prix",      # Night race since 2014
    "Singapore Grand Prix",    # Always night
    "Abu Dhabi Grand Prix",    # Dusk/artificial lighting at Yas Marina
    "Saudi Arabian Grand Prix",
    "Las Vegas Grand Prix",
    "Qatar Grand Prix",
}
df["is_night_race"] = df["EventName"].isin(NIGHT_RACES).astype(int)

# Street circuits (temporary surfaces, walls, different grip characteristics)
STREET_CIRCUITS = {
    "Monaco Grand Prix",
    "Azerbaijan Grand Prix",
    "Singapore Grand Prix",
    "Saudi Arabian Grand Prix",
    "Las Vegas Grand Prix",
    "Miami Grand Prix",
}
df["is_street_circuit"] = df["EventName"].isin(STREET_CIRCUITS).astype(int)

In [ ]:
new_cols = [
    "Q2_appearance","Q3_appearance",
    "Q1_gap","Q2_gap","Q3_gap",
    "season_stage_ratio","driver_career_races","driver_q3_rate_season",
    "driver_circuit_q_pos_hist","prior_year_q_pos_same_circuit",
    "team_rolling_q_pos_5r","driver_rolling_race_pos_5r","teammate_q_gap_season",
    "is_night_race","is_street_circuit",
]
print(f"Shape: {df.shape}")
print(f"
New columns ({len(new_cols)}): {new_cols}")
df[df["Abbreviation"]=="HAM"][new_cols].head(8)

## Points & Championship Context

Estimated from `Position` using the standard 2010+ points system (25-18-15-12-10-8-6-4-2-1). All features are **shifted** to prevent leakage.

In [ ]:
POINTS_MAP = {1:25, 2:18, 3:15, 4:12, 5:10, 6:8, 7:6, 8:4, 9:2, 10:1}
df["race_pts"] = df["Position"].apply(
    lambda p: POINTS_MAP.get(int(p), 0) if pd.notna(p) else 0
)

# Driver cumulative points this season (shifted)
df = df.sort_values(["Season","Round"]).reset_index(drop=True)
df["driver_cum_pts"] = (
    df.groupby(["Season","Abbreviation"])["race_pts"]
    .transform(lambda x: x.shift(1).cumsum().fillna(0))
)

# Team cumulative points this season (both drivers, shifted)
team_rnd = (
    df.groupby(["Season","Round","TeamName"])["race_pts"].sum()
    .reset_index().rename(columns={"race_pts":"_trp"})
    .sort_values(["Season","Round"])
)
team_rnd["team_cum_pts"] = (
    team_rnd.groupby(["Season","TeamName"])["_trp"]
    .transform(lambda x: x.shift(1).cumsum().fillna(0))
)
df = df.merge(team_rnd[["Season","Round","TeamName","team_cum_pts"]],
              on=["Season","Round","TeamName"], how="left")

# Gap to championship leader (negative = behind leader)
df["driver_pts_gap_to_leader"] = (
    df["driver_cum_pts"]
    - df.groupby(["Season","Round"])["driver_cum_pts"].transform("max")
)
df["team_pts_gap_to_leader"] = (
    df["team_cum_pts"]
    - df.groupby(["Season","Round"])["team_cum_pts"].transform("max")
)

## Qualifying vs Race Delta

Places gained between grid and finish. Rolling 5-race average (shifted) captures whether a driver consistently races better or worse than they qualify.

In [ ]:
df["q_vs_race_delta"] = df["GridPosition"] - df["Position"]
df["driver_q_vs_race_delta_5r"] = (
    df.groupby("Abbreviation")["q_vs_race_delta"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

## Circuit Physical Properties

Hardcoded lookup keyed on `EventName`. Altitude matters for power unit output (Mexico City at 2240m is the most extreme). DRS zones and corner count capture the aero vs mechanical grip balance.

In [ ]:
CIRCUIT_PROPS = {
    # EventName: (altitude_m, length_km, num_corners, num_drs_zones)
    "Australian Grand Prix":       (  13, 5.278, 16, 4),
    "Bahrain Grand Prix":          (   7, 5.412, 15, 3),
    "Sakhir Grand Prix":           (   7, 3.543,  7, 2),  # outer layout 2020
    "Chinese Grand Prix":          (   4, 5.451, 16, 2),
    "Spanish Grand Prix":          (  87, 4.655, 16, 2),
    "Monaco Grand Prix":           (   7, 3.337, 19, 1),
    "Canadian Grand Prix":         (  23, 4.361, 14, 2),
    "Austrian Grand Prix":         ( 693, 4.318, 10, 3),
    "Styrian Grand Prix":          ( 693, 4.318, 10, 3),  # same venue
    "British Grand Prix":          ( 126, 5.891, 18, 2),
    "70th Anniversary Grand Prix": ( 126, 5.891, 18, 2),  # same venue
    "Hungarian Grand Prix":        ( 264, 4.381, 14, 1),
    "Belgian Grand Prix":          ( 401, 7.004, 20, 2),
    "Italian Grand Prix":          ( 162, 5.793, 11, 2),
    "Singapore Grand Prix":        (  15, 5.063, 23, 3),
    "Japanese Grand Prix":         (  48, 5.807, 18, 2),
    "Russian Grand Prix":          (   4, 5.853, 18, 2),
    "United States Grand Prix":    ( 159, 5.513, 20, 2),
    "Mexican Grand Prix":          (2240, 4.304, 17, 3),
    "Mexico City Grand Prix":      (2240, 4.304, 17, 3),  # renamed
    "Brazilian Grand Prix":        ( 795, 4.309, 15, 2),
    "São Paulo Grand Prix":        ( 795, 4.309, 15, 2),  # renamed
    "Abu Dhabi Grand Prix":        (   1, 5.281, 16, 3),
    "Malaysian Grand Prix":        (  60, 5.543, 15, 3),
    "German Grand Prix":           ( 117, 4.574, 17, 2),
    "European Grand Prix":         (   0, 6.003, 20, 2),  # Baku 2016
    "Azerbaijan Grand Prix":       (   0, 6.003, 20, 2),
    "French Grand Prix":           ( 422, 5.842, 15, 3),
    "Saudi Arabian Grand Prix":    (   6, 6.174, 27, 3),
    "Qatar Grand Prix":            (  40, 5.419, 16, 2),
    "Miami Grand Prix":            (   2, 5.412, 19, 3),
    "Las Vegas Grand Prix":        ( 645, 6.201, 17, 3),
    "Dutch Grand Prix":            (   3, 4.259, 14, 2),
    "Portuguese Grand Prix":       ( 108, 4.684, 15, 2),
    "Emilia Romagna Grand Prix":   (  21, 4.909, 19, 2),
    "Turkish Grand Prix":          ( 130, 5.338, 14, 2),
    "Eifel Grand Prix":            ( 600, 5.148, 15, 2),
    "Tuscan Grand Prix":           ( 252, 5.245, 15, 3),
}

df["circuit_altitude_m"]  = df["EventName"].map(lambda e: CIRCUIT_PROPS.get(e, (None,)*4)[0])
df["circuit_length_km"]   = df["EventName"].map(lambda e: CIRCUIT_PROPS.get(e, (None,)*4)[1])
df["num_corners"]         = df["EventName"].map(lambda e: CIRCUIT_PROPS.get(e, (None,)*4)[2])
df["num_drs_zones"]       = df["EventName"].map(lambda e: CIRCUIT_PROPS.get(e, (None,)*4)[3])

# Sanity check
missing = df[df["circuit_altitude_m"].isna()]["EventName"].unique()
if len(missing):
    print("WARNING — circuits missing from lookup:", missing)
else:
    print("All circuits mapped OK")

## Power Unit Manufacturer

Engine supplier per team per season. PU performance varies significantly by year and circuit type (altitude, DRS zones).

In [ ]:
def get_pu(team, season):
    if team in ('Mercedes',):
        return 'Mercedes'
    if team in ('Williams', 'Force India', 'Racing Point', 'Aston Martin'):
        return 'Mercedes'
    if team == 'Lotus F1':
        return 'Mercedes'  # switched to Merc for 2015
    if team == 'Manor Marussia':
        return 'Mercedes' if season >= 2016 else 'Ferrari'
    if team in ('Ferrari',):
        return 'Ferrari'
    if team in ('Haas F1 Team',):
        return 'Ferrari'
    if team in ('Sauber', 'Alfa Romeo Racing', 'Alfa Romeo', 'Kick Sauber'):
        return 'Ferrari'
    if team in ('Renault', 'Alpine'):
        return 'Renault'
    if team == 'McLaren':
        if season <= 2017: return 'Honda'
        if season <= 2020: return 'Renault'
        return 'Mercedes'
    if team in ('Red Bull', 'Red Bull Racing'):
        return 'Renault' if season <= 2018 else 'Honda_RBPT'
    if team in ('Toro Rosso', 'AlphaTauri', 'RB', 'Racing Bulls'):
        return 'Renault' if season <= 2017 else 'Honda_RBPT'
    return 'Unknown'

df["power_unit"] = df.apply(lambda r: get_pu(r["TeamName"], r["Season"]), axis=1)
print(df.groupby(["Season","power_unit"])["Abbreviation"].count().unstack(fill_value=0))

## Driver Information

`driver_age`: approximate age at race weekend. `is_home_race`: 1 if driver is racing in their home country GP.

In [ ]:
DRIVER_BIRTH_YEAR = {
    'ALB': 1996, 'ALO': 1981, 'AIT': 1995, 'ANT': 1991,
    'BEA': 1980, 'BOR': 1985, 'BOT': 1989, 'BUT': 1980,
    'COL': 1984, 'DEV': 1995, 'DIR': 1992, 'DOO': 2003,
    'ERI': 1990, 'FIT': 1980, 'GAS': 1996, 'GIO': 1993,
    'GRO': 1986, 'GUT': 1991, 'HAD': 2004, 'HAM': 1985,
    'HAR': 1989, 'HUL': 1987, 'KUB': 1984, 'KVY': 1994,
    'LAT': 1995, 'LAW': 2002, 'LEC': 1997, 'MAG': 1992,
    'MAL': 1985, 'MAS': 1981, 'MAZ': 1999, 'MER': 1991,
    'MSC': 1999, 'NAS': 1990, 'NOR': 2000, 'OCO': 1996,
    'PAL': 1991, 'PER': 1990, 'PIA': 2001, 'RAI': 1979,
    'RIC': 1989, 'ROS': 1985, 'RSS': 1995, 'RUS': 1998,
    'SAI': 1994, 'SAR': 2001, 'SIR': 1995, 'STE': 1993,
    'STR': 1998, 'TSU': 2000, 'VAN': 1992, 'VER': 1997,
    'VET': 1987, 'WEH': 1994, 'ZHO': 1999,
}
df["driver_age"] = df.apply(
    lambda r: r["Season"] - DRIVER_BIRTH_YEAR.get(r["Abbreviation"], np.nan)
    if r["Abbreviation"] in DRIVER_BIRTH_YEAR else np.nan, axis=1
)

# Home race: driver racing in their home-country GP
# Where a GP was renamed (Mexican→Mexico City, Brazilian→São Paulo), both are included
HOME_RACE = {
    'ALO': {'Spanish Grand Prix'},
    'ALB': set(),
    'BOT': set(),
    'BUT': {'British Grand Prix'},
    'DEV': {'Dutch Grand Prix'},
    'ERI': set(),
    'GAS': {'French Grand Prix'},
    'GIO': {'Italian Grand Prix'},
    'GRO': {'French Grand Prix'},
    'HAD': set(),
    'HAM': {'British Grand Prix'},
    'HAR': set(),
    'HUL': {'German Grand Prix'},
    'KVY': {'Russian Grand Prix'},
    'LAT': {'Canadian Grand Prix'},
    'LAW': set(),
    'LEC': {'Monaco Grand Prix'},
    'MAG': set(),
    'MAS': {'Brazilian Grand Prix', 'São Paulo Grand Prix'},
    'MSC': {'German Grand Prix'},
    'NAS': {'Brazilian Grand Prix', 'São Paulo Grand Prix'},
    'NOR': {'British Grand Prix'},
    'OCO': {'French Grand Prix'},
    'PER': {'Mexican Grand Prix', 'Mexico City Grand Prix'},
    'PIA': {'Australian Grand Prix'},
    'RAI': set(),
    'RIC': {'Australian Grand Prix'},
    'ROS': {'German Grand Prix'},
    'RUS': {'British Grand Prix'},
    'SAI': {'Spanish Grand Prix'},
    'SIR': {'Russian Grand Prix'},
    'STR': {'Canadian Grand Prix'},
    'TSU': {'Japanese Grand Prix'},
    'VAN': {'Belgian Grand Prix'},
    'VER': {'Dutch Grand Prix'},
    'VET': {'German Grand Prix'},
    'ZHO': {'Chinese Grand Prix'},
}
df["is_home_race"] = df.apply(
    lambda r: int(r["EventName"] in HOME_RACE.get(r["Abbreviation"], set())), axis=1
)
print("driver_age nulls:", df["driver_age"].isna().sum())
print("home races flagged:", df["is_home_race"].sum())

## Practice Session Gaps (OpenF1 — 2023–2025)

Best lap gap to session fastest per driver in FP1/FP2/FP3. Fetched via [OpenF1 API](https://openf1.org) and cached to `fp_cache.csv`. Coverage: **2023–2025** (OpenF1 full history starts 2023). Rows for 2015–2022 will be `NaN` — handled natively by tree models. **Same-weekend features**: only valid for same-weekend prediction.

In [ ]:
import requests, os, time
import pandas as pd

OPENF1  = "https://api.openf1.org/v1"
FP_CACHE = r'C:\Users\CLL\OneDrive\Documents\GitHub\F1-Race-Predictor\notebooks\fp_cache.csv'

def _get(endpoint, params=None, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = requests.get(f"{OPENF1}/{endpoint}", params=params, timeout=30)
            if r.status_code == 429:
                wait = 20 * (attempt + 1)
                print(f"    429 rate-limit — sleeping {wait}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            time.sleep(0.4)
            return r.json()
        except requests.exceptions.HTTPError as e:
            if '429' in str(e):
                time.sleep(20 * (attempt + 1)); continue
            raise
    raise RuntimeError(f"Max retries exceeded: {endpoint}")

def _save_fp(records):
    pd.DataFrame(records).to_csv(FP_CACHE, index=False)

def fetch_fp_openf1(years=(2023, 2024, 2025)):
    existing = pd.read_csv(FP_CACHE) if os.path.exists(FP_CACHE) else pd.DataFrame()
    done = set()
    if not existing.empty:
        done = set(zip(existing["Season"].astype(int),
                       existing["EventName"], existing["fp_session"]))
    records = list(existing.to_dict("records")) if not existing.empty else []

    for year in years:
        meetings = [m for m in _get("meetings", {"year": year})
                    if "Testing" not in m.get("meeting_name", "")]
        for meeting in meetings:
            mk, mname = meeting["meeting_key"], meeting["meeting_name"]
            for fp_label, fp_api in [("FP1","Practice 1"),("FP2","Practice 2"),("FP3","Practice 3")]:
                if (year, mname, fp_label) in done:
                    continue
                try:
                    sessions = _get("sessions", {"meeting_key": mk, "session_name": fp_api})
                    if not sessions:
                        done.add((year, mname, fp_label)); continue
                    sk = sessions[0]["session_key"]
                    laps_raw  = _get("laps",    {"session_key": sk})
                    if not laps_raw:
                        done.add((year, mname, fp_label)); continue
                    drivers_raw = _get("drivers", {"session_key": sk})
                    drv_map = {d["driver_number"]: d.get("name_acronym", f"D{d['driver_number']}")
                               for d in drivers_raw}
                    laps_df = pd.DataFrame(laps_raw)
                    valid = laps_df[laps_df["lap_duration"].notna() & ~laps_df["is_pit_out_lap"]]
                    if valid.empty:
                        done.add((year, mname, fp_label)); continue
                    best  = valid.groupby("driver_number")["lap_duration"].min()
                    s_min = best.min()
                    for drv_num, t in best.items():
                        records.append({"Season": year, "EventName": mname,
                                        "Abbreviation": drv_map.get(drv_num, f"D{drv_num}"),
                                        "fp_session": fp_label,
                                        "fp_gap": round(t - s_min, 3)})
                    done.add((year, mname, fp_label))
                    _save_fp(records)   # incremental save
                    print(f"  OK: {mname} {fp_label} ({len(best)} drivers)")
                except requests.exceptions.HTTPError as e:
                    if e.response is not None and e.response.status_code == 404:
                        done.add((year, mname, fp_label))   # no session (sprint)
                    else:
                        print(f"  ERR: {mname} {fp_label}: {e}")
                except Exception as e:
                    print(f"  ERR: {mname} {fp_label}: {type(e).__name__}: {e}")
        print(f"  {year} done — cache: {len(records)} records")

    _save_fp(records)
    return pd.DataFrame(records)

print("Fetching FP data (OpenF1, 2023-2025) — resumes from cache if interrupted...")
fp_df = fetch_fp_openf1()
print(f"FP records loaded: {len(fp_df)}")

In [ ]:
# Merge FP gaps (one column per session)
for fp_label in ("FP1", "FP2", "FP3"):
    sub = (
        fp_df[fp_df["fp_session"] == fp_label]
        [["Season", "EventName", "Abbreviation", "fp_gap"]]
        .rename(columns={"fp_gap": f"{fp_label.lower()}_gap"})
    )
    df = df.merge(sub, on=["Season", "EventName", "Abbreviation"], how="left")

fp_cols = ["fp1_gap", "fp2_gap", "fp3_gap"]
coverage = df[fp_cols].notna().mean().round(3)
print("FP coverage by column:")
print(coverage.to_string())
print(f"\n2023+ rows: {(df['Season'] >= 2023).sum()} | "
      f"FP3 coverage 2023+: {df.loc[df['Season']>=2023, 'fp3_gap'].notna().mean():.1%}")

## Weather at Qualifying (OpenF1 — 2023–2025)

Track/air temperature, humidity, wind speed, and rain flag at qualifying. Cached to `weather_cache.csv`. Same coverage as FP data (2023–2025 complete).

In [ ]:
WEATHER_CACHE = r'C:\Users\CLL\OneDrive\Documents\GitHub\F1-Race-Predictor\notebooks\weather_cache.csv'

def _save_weather(records):
    pd.DataFrame(records).to_csv(WEATHER_CACHE, index=False)

def fetch_weather_openf1(years=(2023, 2024, 2025)):
    existing = pd.read_csv(WEATHER_CACHE) if os.path.exists(WEATHER_CACHE) else pd.DataFrame()
    done = set()
    if not existing.empty:
        done = set(zip(existing["Season"].astype(int), existing["EventName"]))
    records = list(existing.to_dict("records")) if not existing.empty else []

    for year in years:
        meetings = [m for m in _get("meetings", {"year": year})
                    if "Testing" not in m.get("meeting_name", "")]
        for meeting in meetings:
            mk, mname = meeting["meeting_key"], meeting["meeting_name"]
            if (year, mname) in done:
                continue
            try:
                sessions = _get("sessions", {"meeting_key": mk, "session_name": "Qualifying"})
                if not sessions:
                    done.add((year, mname)); continue
                sk = sessions[0]["session_key"]
                weather_raw = _get("weather", {"session_key": sk})
                if not weather_raw:
                    done.add((year, mname)); continue
                wdf = pd.DataFrame(weather_raw)
                records.append({
                    "Season": year, "EventName": mname,
                    "is_wet_qualifying":  int(wdf["rainfall"].gt(0).any()),
                    "track_temp_avg":     round(wdf["track_temperature"].mean(), 1),
                    "air_temp_avg":       round(wdf["air_temperature"].mean(), 1),
                    "humidity_avg":       round(wdf["humidity"].mean(), 1),
                    "wind_speed_avg":     round(wdf["wind_speed"].mean(), 2),
                })
                done.add((year, mname))
                _save_weather(records)  # incremental save
                print(f"  OK: {mname} wet={records[-1]['is_wet_qualifying']} {records[-1]['track_temp_avg']}°C")
            except requests.exceptions.HTTPError as e:
                if e.response is not None and e.response.status_code == 404:
                    done.add((year, mname))
                else:
                    print(f"  ERR: {mname}: {e}")
            except Exception as e:
                print(f"  ERR: {mname}: {type(e).__name__}: {e}")
        print(f"  {year} done — cache: {len(records)} records")

    _save_weather(records)
    return pd.DataFrame(records)

print("Fetching qualifying weather (OpenF1, 2023-2025)...")
weather_df = fetch_weather_openf1()
print(f"Weather records: {len(weather_df)}")

In [ ]:
weather_cols = ["is_wet_qualifying", "track_temp_avg", "air_temp_avg",
                "humidity_avg", "wind_speed_avg"]
df = df.merge(weather_df[["Season", "EventName"] + weather_cols],
              on=["Season", "EventName"], how="left")
print("Weather coverage:", df[weather_cols].notna().mean().round(3).to_dict())

In [ ]:
all_features = [
    # Original
    "reg_disruption_index", "years_since_reg_change",
    # Cleaning artifacts
    "Q2_appearance", "Q3_appearance",
    # Gaps
    "Q1_gap", "Q2_gap", "Q3_gap",
    # Rolling/historical
    "season_stage_ratio", "driver_career_races", "driver_q3_rate_season",
    "driver_circuit_q_pos_hist", "prior_year_q_pos_same_circuit",
    "team_rolling_q_pos_5r", "driver_rolling_race_pos_5r", "teammate_q_gap_season",
    # Static lookups
    "is_night_race", "is_street_circuit",
    # Points & championship
    "driver_cum_pts", "team_cum_pts",
    "driver_pts_gap_to_leader", "team_pts_gap_to_leader",
    # Delta
    "driver_q_vs_race_delta_5r",
    # Circuit
    "circuit_altitude_m", "circuit_length_km", "num_corners", "num_drs_zones",
    # PU & driver
    "power_unit", "driver_age", "is_home_race",
    # FP (same-weekend)
    "fp1_gap", "fp2_gap", "fp3_gap",
    # Weather (same-weekend)
    "is_wet_qualifying", "track_temp_avg", "air_temp_avg", "humidity_avg", "wind_speed_avg",
]
print(f"Total features (excl. target): {len(all_features)}")
print(f"DataFrame shape: {df.shape}")
missing = [c for c in all_features if c not in df.columns]
print(f"Missing from df: {missing}")
df[all_features].describe(include='all').T[["count","mean","std","min","max"]].round(2)

In [12]:
df.loc[(df['Abbreviation'] == 'GAS')]

,Season,Round,EventName,Abbreviation,TeamName,GridPosition,Q1_s,Q2_s,Q3_s,Position,reg_disruption_index,years_since_reg_change
1133,2017,15,Malaysian Grand Prix,GAS,Toro Rosso,15.0,92.547,92.558,9999.000,14.0,2,3
1152,2017,16,Japanese Grand Prix,GAS,Toro Rosso,14.0,91.317,9999.000,9999.000,13.0,2,3
1192,2017,18,Mexican Grand Prix,GAS,Toro Rosso,20.0,0.000,0.000,0.000,13.0,2,3
1211,2017,19,Brazilian Grand Prix,GAS,Toro Rosso,19.0,70.686,9999.000,9999.000,12.0,2,3
1235,2017,20,Abu Dhabi Grand Prix,GAS,Toro Rosso,17.0,99.724,9999.000,9999.000,16.0,2,3
...,...,...,...,...,...,...,...,...,...,...,...,...
4378,2025,8,Monaco Grand Prix,GAS,Alpine,17.0,71.994,9999.000,9999.000,20.0,0,3
4386,2025,9,Spanish Grand Prix,GAS,Alpine,8.0,73.081,72.611,72.199,8.0,0,3
4412,2025,10,Canadian Grand Prix,GAS,Alpine,20.0,72.667,9999.000,9999.000,15.0,0,3
4430,2025,11,Austrian Grand Prix,GAS,Alpine,10.0,65.054,64.846,65.649,13.0,0,3
